# Image Creator tour
A hands-on tour of the Image Creator stage (`galactic_tidal_constraints.simr.image_creator`, imported as `img`, MDRN-3 step 1): it turns one
particle set into one image, with the two galaxies' centers on pixels the config chooses, and maps any positions
(particles or an orbit) into the same pixels. Run the cells in order.

`simr` doesn't run it yet (step 2), so here it's called directly on `sim`'s arrays. Design: `docs/design/image_creator.md`.

In [ ]:
%matplotlib inline
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import image_creator as img
from galactic_tidal_constraints.simr import simulation as sim
from galactic_tidal_constraints.simr.image_creator import ImageConfig

## A model and its arrays
Model SDSS 587722984435351614 (the first line of JSPAM's `archive/data.txt`). Three arrays from `sim`:
- `particles`: every particle at the end of the run, `(n1 + n2 + 1, 6)`; the last row is the secondary's center
- `unperturbed`: the same disks placed at today's centers, with no steps
- `orbit`: the secondary's center at every step, `(n_steps + 1, 7)`

`n1` and `n2` differ on purpose: the image creator splits the rows with both.

In [ ]:
params = np.array([-9.93853, -4.5805, 15.43348, -1.54025, -2.97899, 4.69234, 34.41671, 29.49607, 2.97913, 4.2501,
                   301.0, 35.5, 310.80745, 321.98757])
sim_config = sim.SimulatorConfig(n1=4000, n2=3000)
particles = sim.simulate_particles(params, sim_config).astype(np.float32)
unperturbed = sim.unperturbed_particles(params, sim_config).astype(np.float32)
orbit = sim.simulate_orbit(params, sim_config).astype(np.float32)
print("particles", particles.shape, " unperturbed", unperturbed.shape, " orbit", orbit.shape)
print("secondary in the model:      ", params[:3])
print("secondary at the run's end:  ", particles[-1, :3], " (1-2 steps past now)")

## The config and the frame
`ImageConfig` sets the image size and the pixels the two centers land on, as `(column, row)`. Integer positions
are pixel centers, as in `plt.imshow`, and rows count down (the sky's +y is up in the picture).

`frame_matrix` is one `(2, 4)` matrix per model, built from the model's parameters only: view rotation (the
identity: look down z onto x–y) → drop z → rotate and scale in the plane → flip y → shift. `to_pixels` applies it
to any positions, so the primary (the origin) and the secondary (the model's `x, y, z`) land exactly on their pixels.

In [ ]:
config = ImageConfig(
    primary_center=(32, 64),
    secondary_center=(96, 64),
    brightness="log",
)
print(config)
print("frame matrix:\n", img.frame_matrix(params, config).round(4))
print("centers in pixels:\n", img.to_pixels(np.array([[0, 0, 0], params[:3]]), params, config))

## Particles and orbit together
The same frame maps the orbit, so it plots straight over the particles or the image. The orbit ends at the
secondary's pixel (to within the 1–2 steps a run goes past now).

In [ ]:
p_px = img.to_pixels(particles, params, config)
o_px = img.to_pixels(orbit, params, config)
model_image = img.create_image(particles, params, sim_config.n1, sim_config.n2, config)

fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
n1 = sim_config.n1
ax[0].scatter(*p_px[:n1].T, s=0.3, label="primary")
ax[0].scatter(*p_px[n1:-1].T, s=0.3, label="secondary")
ax[0].set_aspect("equal")
ax[0].set_title("particles in pixels")
ax[0].legend(markerscale=10)
ax[1].imshow(model_image, cmap="gray")
ax[1].set_title("model image")
for a in ax:
    a.plot(*o_px.T, color="tab:red", lw=1)
    a.plot(*np.array([config.primary_center, config.secondary_center]).T, "x", color="tab:orange", ms=8)
    # Keep the image's frame: the orbit runs far outside it, and plotting it would widen the axes
    a.set_xlim(-0.5, config.width - 0.5)
    a.set_ylim(config.height - 0.5, -0.5)  # rows count down, as in an image
plt.show()

## Model and unperturbed images
Both share the model's frame exactly, so they line up pixel for pixel.

In [ ]:
unperturbed_image = img.create_image(unperturbed, params, sim_config.n1, sim_config.n2, config)
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(model_image, cmap="gray")
ax[0].set_title("model")
ax[1].imshow(unperturbed_image, cmap="gray")
ax[1].set_title("unperturbed")
ax[2].imshow(model_image - unperturbed_image, cmap="RdBu_r", vmin=-1, vmax=1)
ax[2].set_title("model - unperturbed")
plt.show()
print(model_image.dtype, model_image.shape, model_image.min(), model_image.max())

## Blur and brightness
`blur_sigma` is in pixels (0 = none); the kernel reaches `blur_truncate` sigmas each side (4, scipy's default).
`brightness="log"` uses log(1 + counts), which brings out faint tails. Both come before the divide by the max.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 10))
for row, brightness in enumerate(["linear", "log"]):
    for col, sigma in enumerate([0.0, 0.5, 1.0]):
        settings = replace(config, blur_sigma=sigma, brightness=brightness)  # config's centers and size
        ax[row, col].imshow(img.create_image(particles, params, sim_config.n1, sim_config.n2, settings), cmap="gray")
        ax[row, col].set_title(f"{brightness}, blur_sigma {sigma}")
plt.show()

## Zooming in
Centers may lie outside the image: put the primary in the middle and the secondary far off to the right, and the
primary fills the frame. Here the two are 4x farther apart than in `config`, so everything is 4x bigger.

In [ ]:
apart = np.subtract(config.secondary_center, config.primary_center)  # config's separation, in pixels
middle = np.array([config.width / 2, config.height / 2])
zoom = replace(config, primary_center=tuple(middle), secondary_center=tuple(middle + 4 * apart), blur_sigma=1.0)
zoomed = img.create_image(particles, params, sim_config.n1, sim_config.n2, zoom)
plt.imshow(zoomed, cmap="gray")
plt.plot(*img.to_pixels(orbit, params, zoom).T, color="tab:red", lw=1)
plt.xlim(-0.5, zoom.width - 0.5)
plt.ylim(zoom.height - 0.5, -0.5)
plt.title("zoomed on the primary")
plt.show()

## What fails
- A config with two equal centers (no scale fits): an error when the config is made.
- Particles whose rows don't number `n1 + n2 + 1`, or whose last row isn't the model's secondary (within 10%).
- A secondary right behind the primary (x = y = 0): no frame.

In [ ]:
attempts = {
    "equal centers": lambda: ImageConfig(primary_center=(50, 50), secondary_center=(50, 50)),
    "wrong n2": lambda: img.create_image(particles, params, sim_config.n1, sim_config.n2 + 1),
    "rows reversed": lambda: img.create_image(particles[::-1], params, sim_config.n1, sim_config.n2),
    "secondary behind": lambda: img.frame_matrix(np.r_[0.0, 0.0, 5.0, params[3:]]),
}
for name, attempt in attempts.items():
    try:
        attempt()
    except ValueError as error:
        print(f"{name}: ValueError: {error}")

## Saving a PNG to look at
`to_png` writes an 8-bit grayscale PNG (standard library only). Images stay `float32` in [0, 1] everywhere else.
`output/` is ignored by git.

In [ ]:
folder = Path("output")
folder.mkdir(exist_ok=True)
img.to_png(model_image, folder / "image_tour_model.png")
img.to_png(zoomed, folder / "image_tour_zoomed.png")
print(sorted(p.name for p in folder.iterdir()))

## From `simr.run` (step 2)
`simr` makes images for many models at once. The request gets an `image` section, like `simulator`: one
entry or a list (centers as lists, as JSON writes them), and `run.return` can name `image.model` and
`image.unperturbed`. Images are indexed `[model, simulator, image]`. `simr` computes the particles an image
needs even when they aren't returned. Images can't be saved yet (step 3).

In [ ]:
from galactic_tidal_constraints.simr import simr

heavier = params.copy()
heavier[sim.PARAM_INDEX["mass2"]] *= 2
models = np.stack([params, heavier])
request = {
    "run": {"return": ["sim.orbit", "image.model", "image.unperturbed"]},
    "simulator": {"n1": sim_config.n1, "n2": sim_config.n2},
    "image": [
        {"primary_center": list(config.primary_center), "secondary_center": list(config.secondary_center),
         "brightness": "log"},
        {"primary_center": list(config.primary_center), "secondary_center": list(config.secondary_center),
         "brightness": "log", "blur_sigma": 0.75},
    ],
}
result = simr.run(models, request)
print("image.model:", result.data["image.model"].shape, " [model, simulator, image]")
print("sim status:  ", result.status["sim"].tolist())
print("image status:", result.status["image"].tolist())
print("image entries:", result.images)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 10))
for m in range(2):
    o_px = img.to_pixels(result.data["sim.orbit"][m, 0], models[m], result.images[0])
    for col, (name, i) in enumerate([("image.model", 0), ("image.model", 1), ("image.unperturbed", 1)]):
        a = ax[m, col]
        a.imshow(result.data[name][m, 0, i], cmap="gray")
        a.plot(*o_px.T, color="tab:red", lw=0.8)
        a.set_xlim(-0.5, result.images[i].width - 0.5)
        a.set_ylim(result.images[i].height - 0.5, -0.5)
        a.set_title(f"model {m}: {name}, image entry {i}")
plt.show()